# DAISY 3 Full-Book Generation in Colab

This notebook builds the full DAISY 3 package for the Vietnamese EPUB in Google Colab.

Run the cells in order.
- Cell 1: mount Drive
- Cell 2: confirm EPUB and repo paths
- Cell 3: install ffmpeg and Python deps
- Cell 4: clone/update repo
- Cell 5: read Azure credentials
- Cell 6: dry run without Azure cost
- Cell 7: full build with Azure
- Cell 8: verify output

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✓ Drive mounted')

In [ ]:
import os

DRIVE_BASE = '/content/drive/MyDrive/daisy-psychology-of-money'
SOURCE_EPUB = f'{DRIVE_BASE}/data/tam-ly-hoc-ve-tien.epub'
DRIVE_OUTPUT = f'{DRIVE_BASE}/output'
WORKDIR = '/content/Daisy'
REPO_URL = 'https://github.com/tthongbos/Daisy.git'
REPO_REF = 'main'

if not os.path.exists(SOURCE_EPUB):
    raise FileNotFoundError(f'EPUB not found at {SOURCE_EPUB}')
print(f'✓ EPUB found: {SOURCE_EPUB}')
print(f'  output dir: {DRIVE_OUTPUT}')
print(f'  repo dir: {WORKDIR}')

In [ ]:
import subprocess
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], check=True)
print('✓ ffmpeg installed')

In [ ]:
import os
import shutil
import subprocess

if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)

result = subprocess.run(['git', 'clone', '-b', REPO_REF, REPO_URL, WORKDIR], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError(result.stderr)
print(f'✓ repo cloned to {WORKDIR}')

In [ ]:
import os
import subprocess

os.chdir(WORKDIR)
result = subprocess.run(['pip', 'install', '-q', '-e', '.'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError(result.stderr)
print('✓ dependencies installed')

In [ ]:
from google.colab import userdata
import os

try:
    azure_key = userdata.get('AZURE_SPEECH_KEY')
    azure_region = userdata.get('AZURE_SPEECH_REGION')
    azure_voice = userdata.get('AZURE_SPEECH_VOICE') or 'vi-VN-HoaiMyNeural'
except Exception:
    azure_key = None
    azure_region = None
    azure_voice = 'vi-VN-HoaiMyNeural'

if azure_key and azure_region:
    os.environ['AZURE_SPEECH_KEY'] = azure_key
    os.environ['AZURE_SPEECH_REGION'] = azure_region
    os.environ['AZURE_SPEECH_VOICE'] = azure_voice
    print('✓ Azure credentials loaded')
    print(f'  region: {azure_region}')
    print(f'  voice: {azure_voice}')
else:
    print('⚠ Azure credentials missing. Dry-run only works without them.')

In [ ]:
import os
import subprocess

os.chdir(WORKDIR)
print('Running dry-run (no Azure cost)...')
result = subprocess.run(['make', 'full-book-dry-run'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f'dry-run failed with exit code {result.returncode}')
print('✓ dry-run complete')

In [ ]:
import os
import subprocess

os.chdir(WORKDIR)
if not os.environ.get('AZURE_SPEECH_KEY') or not os.environ.get('AZURE_SPEECH_REGION'):
    print('⚠ Skipping full build because Azure credentials are not set.')
else:
    print('Running full build...')
    result = subprocess.run(['make', 'full-book'], capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f'full build failed with exit code {result.returncode}')
    print('✓ full build complete')

In [ ]:
import glob
import os
import subprocess

os.chdir(WORKDIR)
print('Verification checklist')
print('=' * 60)

smil_files = glob.glob('build/daisy/*.smil')
mp3_files = glob.glob('build/daisy/*.mp3')
print(f'✓ found {len(smil_files)} SMIL files and {len(mp3_files)} MP3 files')

npt_check = subprocess.run("grep -r 'npt=' build/daisy/*.smil 2>/dev/null || true", shell=True, capture_output=True, text=True)
if npt_check.stdout.strip():
    raise RuntimeError(f'Found npt= in generated SMIL: {npt_check.stdout}')
print('✓ no npt= found in SMIL files')

result = subprocess.run(['python', '-m', 'daisy_book.validate_daisy', '--input', 'build/daisy'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f'validation failed with exit code {result.returncode}')
print('✓ validation passed')